In [22]:
import pandas as pd
from scipy import stats

# Part 1 – Quantifying Missing Data

In [23]:
netflix_titles = pd.read_csv("netflix_titles.csv")
print(netflix_titles.shape)

(8807, 12)


### Missing Value Audit:

In [24]:
missing_counts = netflix_titles.isnull().sum()

missing_counts

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

### Impact Scaling: 

In [25]:
missing_percentages = (
    netflix_titles.isnull().sum()
    / len(netflix_titles)
    * 100
)

missing_percentages.sort_values(ascending=False)

director        29.908028
country          9.435676
cast             9.367549
date_added       0.113546
rating           0.045418
duration         0.034064
show_id          0.000000
type             0.000000
title            0.000000
release_year     0.000000
listed_in        0.000000
description      0.000000
dtype: float64

The director column contains the largest amount of missing data, followed by country and cast. These columns have the greatest impact on data quality because a significant portion of rows contain incomplete information.

In [26]:
# Part 2 – Cleaning Strategies (Dropping vs. Imputing)

In [27]:
## Strategy A (Imputation):

In [28]:
netflix_titles["director"] = netflix_titles["director"].fillna("Unknown")

netflix_titles["director"].isnull().sum()

np.int64(0)

## Strategy B (Row Dropping):

In [29]:
dropped_df = netflix_titles.dropna(
    subset=["director", "cast"]
)


## Reflection:

In [30]:
print("Original shape:", netflix_titles.shape)
print("Dropped shape:", dropped_df.shape)

Original shape: (8807, 12)
Dropped shape: (7982, 12)


## Rows lost

In [31]:
rows_lost = len(netflix_titles) - len(dropped_df)

print("Rows lost:", rows_lost)

Rows lost: 825


After removing rows with missing values in either the director or cast columns, a total of 825 rows were lost. Losing this much data could significantly affect an analysis because it reduces the available sample size and may remove important records. For example, if the missing values are concentrated in a specific genre or region, the results could become biased.

# Part 3 – Processing Datetime Data

In [32]:
## Data Type Verification

In [33]:
print(netflix_titles["date_added"].dtype)

str


## Datetime Conversion:

In [34]:
netflix_titles["date_added"] = netflix_titles["date_added"].str.strip()

In [35]:
netflix_titles["date_added"] = pd.to_datetime(
    netflix_titles["date_added"]
)

In [36]:
netflix_titles.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      8807 non-null   str           
 4   cast          7982 non-null   str           
 5   country       7976 non-null   str           
 6   date_added    8797 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8803 non-null   str           
 9   duration      8804 non-null   str           
 10  listed_in     8807 non-null   str           
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), str(10)
memory usage: 825.8 KB


## Feature Extraction:

In [37]:
netflix_titles["day_added"] = (
    netflix_titles["date_added"]
    .dt.day
)

In [38]:
netflix_titles.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,day_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",25.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",24.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,24.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",24.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,24.0


# Formulating and Testing Hypotheses

In [39]:
movies = netflix_titles[
    netflix_titles["type"] == "Movie"
]["release_year"]

tv_shows = netflix_titles[
    netflix_titles["type"] == "TV Show"
]["release_year"]

In [41]:
t_stat, p_value = stats.ttest_ind(
    movies,
    tv_shows,
    equal_var=False
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: -20.976297477593885
P-value: 3.7115400804498955e-95
